In [111]:
import pandas as pd
from pathlib import Path

events_path = Path("../data/processed/winger_events.parquet")

df = pd.read_parquet(events_path)

df.shape

(2656669, 15)

In [112]:
dribbles = (
    df[df["type.name"] == "Dribble"]
    .groupby("player_id")
    .size()
    .rename("dribbles")
)

In [113]:
carries = (
    df[df["type.name"] == "Carry"]
    .groupby("player_id")
    .size()
    .rename("carries")
)


In [114]:
passes = (
    df[df["type.name"] == "Pass"]
    .groupby("player_id")
    .size()
    .rename("passes")
)

In [115]:
shots = (
    df[df["type.name"] == "Shot"]
    .groupby("player_id")
    .size()
    .rename("shots")
)

In [116]:
xg = (
    df[df["type.name"] == "Shot"]
    .groupby("player_id")["shot.statsbomb_xg"]
    .sum()
    .rename("xg")
)

In [117]:
features = pd.concat(
    [dribbles, carries, passes, shots, xg],
    axis=1
).fillna(0)

features.head()

,dribbles,carries,passes,shots,xg
player_id,,,,,
2937,3.0,13.0,10.0,0.0,0.000000
2941,30.0,236.0,177.0,21.0,3.395412
2944,8.0,87.0,130.0,4.0,0.351674
2948,12.0,101.0,76.0,7.0,1.190467
2950,15.0,350.0,388.0,10.0,1.642851


In [118]:
players = df[["player_id", "player_name"]].dropna().drop_duplicates(subset=["player_id"])
features = features.merge(players, on="player_id", how="left")


In [119]:
# remove old minutes column if it exists
features = features.drop(columns=["minutes"], errors="ignore")

# compute minutes played
minutes = (
    df[["match_id","player_id","start_minute","end_minute"]]
    .drop_duplicates()
)

minutes["minutes"] = (
    minutes["end_minute"].fillna(130) -
    minutes["start_minute"].fillna(0)
)

minutes = (
    minutes.groupby("player_id")["minutes"]
    .sum()
    .reset_index()
)

# merge minutes
features = features.merge(minutes, on="player_id", how="left")

# remove players with very small minutes
features = features[features["minutes"] >= 300]

# compute per90 metrics
for col in ["dribbles","carries","passes","shots","xg"]:
    features[f"{col}_per90"] = (features[col] / features["minutes"]) * 90

features.head()

,player_id,dribbles,carries,passes,shots,xg,player_name,minutes,dribbles_per90,carries_per90,passes_per90,shots_per90,xg_per90
1,2941,30.0,236.0,177.0,21.0,3.395412,Ismaïla Sarr,1040.0,2.596154,20.423077,15.317308,1.817308,0.293834
2,2944,8.0,87.0,130.0,4.0,0.351674,Benjamin Corgnet,1170.0,0.615385,6.692308,10.000000,0.307692,0.027052
3,2948,12.0,101.0,76.0,7.0,1.190467,Nabil Fekir,780.0,1.384615,11.653846,8.769231,0.807692,0.137362
4,2950,15.0,350.0,388.0,10.0,1.642851,Jonas Martin,910.0,1.483516,34.615385,38.373626,0.989011,0.162480
6,2955,72.0,1580.0,2137.0,142.0,12.233072,Ronny Rodelin,4030.0,1.607940,35.285360,47.724566,3.171216,0.273195


In [120]:
features.shape


(1438, 13)

In [121]:
features.columns

Index(['player_id', 'dribbles', 'carries', 'passes', 'shots', 'xg',
       'player_name', 'minutes', 'dribbles_per90', 'carries_per90',
       'passes_per90', 'shots_per90', 'xg_per90'],
      dtype='object')

In [122]:
features.describe()

,player_id,dribbles,carries,passes,shots,xg,minutes,dribbles_per90,carries_per90,passes_per90,shots_per90,xg_per90
count,1438.0,1438.000000,1438.000000,1438.000000,1438.000000,1438.000000,1438.000000,1438.000000,1438.000000,1438.000000,1438.000000,1438.000000
mean,39832.037552,33.716273,370.842142,379.017385,20.821975,2.104630,1450.973574,1.635857,18.824900,19.491765,1.040478,0.103849
std,82069.889881,64.808138,592.717096,587.957753,38.265876,4.738397,1447.442310,1.345069,10.877708,11.493782,0.793545,0.105949
min,2941.0,0.000000,0.000000,3.000000,0.000000,0.000000,390.000000,0.000000,0.000000,0.692308,0.000000,0.000000
25%,6362.0,5.000000,70.000000,72.000000,3.000000,0.217181,520.000000,0.692308,10.615385,10.765385,0.461538,0.029956
50%,10632.5,14.000000,172.000000,172.000000,9.000000,0.824948,910.000000,1.384615,17.538462,18.000000,0.923077,0.075717
75%,26689.75,41.000000,471.500000,484.000000,26.000000,2.552274,2080.000000,2.306351,26.169231,26.307692,1.397596,0.143052
max,416907.0,1320.000000,12186.000000,11555.000000,932.000000,119.681269,22880.000000,10.615385,69.923077,77.538462,5.815385,0.796270


In [123]:
from pathlib import Path

output_path = Path("../data/processed/winger_features.parquet")

output_path.parent.mkdir(parents=True, exist_ok=True)

features.to_parquet(output_path, index=False)

print("Saved to:", output_path)

Saved to: ..\data\processed\winger_features.parquet


In [124]:
features["player_id"].duplicated().sum()

0

In [125]:
features.isna().sum()

player_id         0
dribbles          0
carries           0
passes            0
shots             0
xg                0
player_name       0
minutes           0
dribbles_per90    0
carries_per90     0
passes_per90      0
shots_per90       0
xg_per90          0
dtype: int64

In [126]:
features["minutes"].describe()

count     1438.000000
mean      1450.973574
std       1447.442310
min        390.000000
25%        520.000000
50%        910.000000
75%       2080.000000
max      22880.000000
Name: minutes, dtype: float64

In [127]:
features[
[
"dribbles_per90",
"carries_per90",
"passes_per90",
"shots_per90",
"xg_per90"
]
].describe()

,dribbles_per90,carries_per90,passes_per90,shots_per90,xg_per90
count,1438.000000,1438.000000,1438.000000,1438.000000,1438.000000
mean,1.635857,18.824900,19.491765,1.040478,0.103849
std,1.345069,10.877708,11.493782,0.793545,0.105949
min,0.000000,0.000000,0.692308,0.000000,0.000000
25%,0.692308,10.615385,10.765385,0.461538,0.029956
50%,1.384615,17.538462,18.000000,0.923077,0.075717
75%,2.306351,26.169231,26.307692,1.397596,0.143052
max,10.615385,69.923077,77.538462,5.815385,0.796270
